# SmolUS-RG Training Notebook


**Change for every experiment run**

- **`Experiment_config`**
- **`If data splitting - uncomment the cell`**


**Notebook Flow:**
1. Mount drive and set paths
2. Install dependencies
3. Load code from Drive or GitHub
4. GPU check & configure experiment
5. Initialise experiment tracking e.g. ClearML + timer
6. Create data splits (run once if not yet done)
7. Verify data
8. Load data, model, build trainer
9. Verify collator masking
10. Train
11. Plot training curves
12. Save model + val evaluation
13. Val generation samples
14. Test set evaluation + metrics
15. Efficiency profiling


In [ ]:
# DRIVE_ROOT  — persistent storage (data, checkpoints, outputs)
# REPO_DIR    — the cloned/unzipped pipeline code in Colab's /content

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
from datetime import datetime

N_SAMPLES     = 10 # used for the qualitative analysis of validation set generated text
DRIVE_ROOT = '/content/drive/MyDrive/Experiments/smol_us_rg'
REPO_DIR   = f'/content/smol_us_rg_code'
TIMESTAMP = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f'Drive root : {DRIVE_ROOT}')
print(f'Repo dir   : {REPO_DIR}')


In [ ]:
# Load pipeline code
# Option A — GitHub
# from google.colab import userdata
# GITHUB_TOKEN = userdata.get('TOKEN')
# !git clone https://{GITHUB_TOKEN}@github.com/username/project_name.git {REPO_DIR}

# Option B — Drive zip (default)
!unzip -q {DRIVE_ROOT}/smol_us_rg_code.zip -d /content/
sys.path.insert(0, REPO_DIR)
print('Code path set.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt
print("Dependencies installed.")

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0]) # we use Python: 3.13.15
print("torch :", torch.__version__)   # we use torch : 2.11.0+cu130


In [ ]:
# Experiment tracking - ClearML credentials
from dotenv import load_dotenv
load_dotenv(f'{DRIVE_ROOT}/.env')

from clearml import Task

print('ClearML SDK ready. Connected to:', os.environ.get('CLEARML_WEB_HOST', 'NOT SET'))


In [ ]:
# GPU check + auto-precision
import torch

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    capability    = torch.cuda.get_device_capability()
    auto_precision = 'bf16' if capability[0] >= 8 else 'fp16'
    print(f'Capability   : {capability}')
else:
    auto_precision = 'fp32'

print(f'Precision    : {auto_precision}')


In [ ]:
# Update these paths in the yaml config, then comment out this cell.
# from data.split import run_split

# ANATOMY = 'breast'
# SOURCE_FILE = f'{DRIVE_ROOT}/data/{ANATOMY}_reports.csv'
# OUTPUT_DIR  = f'{DRIVE_ROOT}/data'

# train_path, val_path, test_path = run_split(
#     input_path=SOURCE_FILE,
#     output_dir=OUTPUT_DIR,
#     train_ratio=0.7,
#     val_ratio=0.1,
#     test_ratio=0.2,
#     seed=42,
# )

# print(f'train_path: {train_path}')
# print(f'val_path:   {val_path}')
# print(f'test_path:  {test_path}')

In [ ]:
import shutil, zipfile
from pathlib import Path

# Path to the zipped folder on Drive containing the whole data dir.
# unzips to a folder containing per-anatomy subdirs: data/liver/..., data/thyroid/..., etc.
DATA_ZIP_DRIVE = f"{DRIVE_ROOT}/data.zip"

def _ensure_data_on_ssd(ssd_root: str = "/content/data",
                        zip_drive: str = DATA_ZIP_DRIVE):
    """
    Copy the data.zip from Drive to local SSD and extract it once.
    Subsequent calls (any anatomy) are SSD cache hits
    Drive is touched at most once per runtime, as a single large-file copy.
    """
    ssd_root   = Path(ssd_root)
    marker     = ssd_root / ".data_extracted"
    local_zip  = ssd_root / "data.zip"
    ssd_root.mkdir(parents=True, exist_ok=True)

    if marker.exists():
        return ssd_root

    zip_drive = Path(zip_drive)
    if not zip_drive.exists():
        raise FileNotFoundError(f"data.zip not found on Drive: {zip_drive}")

    print(f"  Copying data.zip Drive → SSD: {zip_drive}")
    shutil.copy2(zip_drive, local_zip)
    print(f"  Extracting {local_zip} → {ssd_root}")
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(ssd_root)
    local_zip.unlink()                                   # reclaim space, keep only images
    marker.touch()                                       # mark complete only after success
    print(f"  Data extracted to SSD.")
    return ssd_root


def stage_images_to_ssd(cfg, ssd_root: str = "/content/data"):
    """
    Ensure the whole data dir is on SSD (copied+extracted from Drive zip,
    once per runtime), then repoint the active anatomy's image_dir at the SSD.
    """
    _ensure_data_on_ssd(ssd_root)

    active  = cfg["dataset"] if cfg["dataset_format"] == "structured" else cfg["csv_dataset"]
    anatomy = cfg["anatomy"]

    # The zip must extract to <ssd_root>/<anatomy>/...
    local_dir = Path(ssd_root) / anatomy
    if not local_dir.exists():
        raise FileNotFoundError(
            f"Anatomy folder not found on SSD: {local_dir}. "
            f"Check the zip's internal layout (expected <anatomy>/... at the top level)."
        )

    n = sum(1 for p in local_dir.rglob('*') if p.is_file())
    print(f"  SSD ready: {local_dir} ({n} files)")
    active["image_dir"] = str(local_dir)
    print(f"  image_dir → {active['image_dir']}")
    return cfg

In [ ]:
def guard_output_dir(cfg: dict, split: str = "test", force: bool = False):
    """
    Returns True if the experiment should run, False if it should be skipped
    because identical results already exist.
    """
    output_dir = Path(cfg["experiment"]["output_dir"])
    metrics_path = output_dir / f"metrics_{split}.json"

    if not metrics_path.exists():
        return True

    # Results exist. Read their metadata and compare to the current cfg.
    try:
        with open(metrics_path, "r", encoding="utf-8") as f:
            prev = json.load(f)
    except Exception:
        # Unreadable/partial file from a crashed run — treat as safe to overwrite.
        print(f"  [guard] {metrics_path} is unreadable; proceeding (will overwrite).")
        return True

    # Fingerprint = the identity fields stored by save_metrics().
    def _fp(d):
        return (
            str(d.get("experiment", "")),
            str(d.get("model_size", "")),
            str(d.get("seed", "")),
            str(d.get("anatomy", "")),
        )

    current = {
        "experiment": cfg["experiment"]["name"],
        "model_size": cfg.get("model_size", ""),
        "seed":       cfg["experiment"].get("seed", ""),
        "anatomy":    cfg.get("anatomy", ""),
    }

    if _fp(prev) == _fp(current):
        # (a) Same experiment, already finished.
        if force:
            print(f"  [guard] Results already exist for '{current['experiment']}' "
                  f"but force=True — re-running and overwriting.")
            return True
        print(f"  [guard] SKIP: '{current['experiment']}' already completed "
              f"(found {metrics_path.name}). Set force=True to re-run.")
        return False

    # (b) Collision: a different experiment already owns this output_dir.
    raise RuntimeError(
        "OUTPUT DIR COLLISION — refusing to overwrite.\n"
        f"  output_dir : {output_dir}\n"
        f"  existing   : experiment={prev.get('experiment')!r}, "
        f"model_size={prev.get('model_size')!r}, seed={prev.get('seed')!r}, "
        f"anatomy={prev.get('anatomy')!r}\n"
        f"  current    : experiment={current['experiment']!r}, "
        f"model_size={current['model_size']!r}, seed={current['seed']!r}, "
        f"anatomy={current['anatomy']!r}\n"
        "Two different experiments resolve to the SAME directory. Give this run a "
        "unique experiment.name (e.g. include model_size in the override)."
    )

In [ ]:
"""
RUN MULTIPLE EXPERIMENTS ONE AFTER THE OTHER
Usage:
    Define an experiment queue in EXPERIMENT_QUEUE below.
    Run this one cell. All experiments run sequentially without intervention.
    Each experiment saves results to Drive and saves to excel.

Failure handling:
    If one experiment crashes, the error is logged, cleans up
    GPU memory, and moves to the next experiment automatically.
    The failed experiment is recorded with status="Failed".
"""

import gc
import json
import os
import random
import subprocess
import sys
import torch
import traceback
from datetime import datetime
from pathlib import Path
from config.loader import load_config

# The main yaml config file sets all experiment variables.
# But each dict here overrides only what changes from the main config experiment variables.
EXPERIMENT_QUEUE = [
  {
    "exp_id":      "EXP-000",
    "config_path": "config/lora_ft.yaml",
    "description": "liver_256_M2W",
    "overrides": {
        "experiment.name": "liver_256_M2W",
        "anatomy": 'liver',
        "model.init_strategy":"huggingface",
        "peft.method": "M2",
        "model_size": "256M",
        "training.num_train_epochs": 4,
        # "force_rerun": False,
        # "checkpointing.auto_resume": True,
    },
  },
]

TRACKER_PATH  = f"{DRIVE_ROOT}/smol_us_rg_experiment_tracker.xlsx"

# free GPU memory between experiments
def _cleanup_gpu(model=None, trainer=None):
    try:
        if trainer is not None:
            del trainer
        if model is not None:
            del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            free_gb = (torch.cuda.get_device_properties(0).total_memory
                       - torch.cuda.memory_allocated()) / 1e9
            print(f"  GPU memory freed. Available: {free_gb:.1f} GB")
    except Exception as e:
        print(f"  Warning: GPU cleanup error: {e}")


# run one complete experiment
def run_experiment(exp_id: str, config_path: str, description: str,
                   overrides: dict = None):
    """
    Runs the full pipeline for one experiment:
        config load → data → model → train → eval (val+test) → efficiency → save
    Returns True on success, False on failure.
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\n{'='*70}")
    print(f"  STARTING: {exp_id}")
    print(f"  Config  : {config_path}")
    print(f"  Desc    : {description}")
    print(f"  Time    : {timestamp}")
    print(f"{'='*70}\n")

    model, trainer, clearml_task, timer = None, None, None, None

    try:
        experiment_config = os.path.join(REPO_DIR, config_path)
        cfg = load_config(experiment_config, drive_root=DRIVE_ROOT, overrides=overrides)

        cfg['training']['precision'] = auto_precision
        if cfg['peft']['mode'] == 'qlora':
            cfg['training']['precision'] = 'fp16'
            cfg['peft']['qlora']['bnb_4bit_compute_dtype'] = 'float16'

        if (cfg['training']['precision'] == 'fp16'
                and cfg['training'].get('optim') == 'adamw_torch_fused'):
            cfg['training']['optim'] = 'adamw_torch'

        cfg = stage_images_to_ssd(cfg)

        # Skip if already done, raise if a different experiment owns this dir.
        force = bool(overrides.get("force_rerun", False)) if overrides else False
        if not guard_output_dir(cfg, split="test", force=force):
            print(f"  {exp_id} skipped — results already present.")
            return True

        exp_name = cfg['experiment']['name']
        output_dir = cfg['experiment']['output_dir']
        print(f"  Experiment name : {exp_name}")
        print(f"  output_dir      : {output_dir}")
        print(f"  PEFT mode       : {cfg['peft']['mode']}")
        print(f"  Precision       : {cfg['training']['precision']}")
        print(f"  Optimizer       : {cfg['training']['optim']}")

        assert exp_name in output_dir, (
            f"output_dir does not contain experiment name!\n"
            f"  name      : {exp_name}\n"
            f"  output_dir: {output_dir}\n"
            f"This means artifacts from different experiments will overwrite each other."
        )

        from utils.clearml_integration import (
            clearml_init, log_dataset_stats, close_task,
            ExperimentTimer, log_timing_to_clearml,
        )
        timer        = ExperimentTimer()
        clearml_task = clearml_init(cfg)
        clearml_task_id = clearml_task.id if clearml_task else ""
        print(f"  ClearML task ID : {clearml_task_id}")

        from data.dataset import load_split_datasets
        timer.start('data_loading')
        train_ds, val_ds = load_split_datasets(cfg)
        timer.stop('data_loading')
        log_dataset_stats(clearml_task, len(train_ds), len(val_ds))
        print(f"  Data: {len(train_ds)} train / {len(val_ds)} val samples")

        from data.dataset import cap_majority_class

        print(f"Before capping: {len(train_ds)} samples")
        train_ds = cap_majority_class(train_ds, max_fraction=0.03)
        print(f"After capping:  {len(train_ds)} samples")

        from collections import Counter
        train_reports = [s["messages"][1]["content"][0]["text"] for s in train_ds]
        counter       = Counter(train_reports)
        dominant_report = counter.most_common(1)[0][0]
        print(f"Uniqueness after capping: {100*len(set(train_reports))/len(train_reports):.1f}%")
        print(f"Top report now appears:   {counter.most_common(1)[0][1]}×")

        # Check the dataset
        train_reports = [s['messages'][1]['content'][0]['text'] for s in train_ds]
        print(f"Total training samples: {len(train_reports)}")
        print(f"Unique reports: {len(set(train_reports))}")
        print(f"Uniqueness ratio: {len(set(train_reports))/len(train_reports):.2%}")

        from collections import Counter
        counter = Counter(train_reports)
        print(f"\nTop 3 most common reports:")
        for text, count in counter.most_common(3):
            print(f"  ({count}x) {text[:100]}")

        from models.loader import load_model_and_processor
        timer.start('model_loading')
        model, processor, lora_config = load_model_and_processor(cfg)
        timer.stop('model_loading')

        processor.image_processor.size = {'longest_edge': 512}
        proc_save_path = os.path.join(cfg['experiment']['output_dir'], 'processor')
        os.makedirs(proc_save_path, exist_ok=True)
        processor.save_pretrained(proc_save_path)

        # Build trainer
        from training.trainer import build_trainer, run_training, save_final_model
        from evaluation.metrics import build_compute_metrics, GenerationExamplesCallback
        from utils.checkpointing import should_resume, build_callbacks

        callbacks = build_callbacks(cfg)
        if cfg.get("evaluation", {}).get("generation_samples", 0) > 0:
          callbacks.append(GenerationExamplesCallback(
              model=model, processor=processor, val_dataset=val_ds, cfg=cfg))
        compute_metrics = build_compute_metrics(processor, cfg)
        trainer = build_trainer(
            model=model, processor=processor,
            train_dataset=train_ds, val_dataset=val_ds,
            lora_config=lora_config, cfg=cfg,
            callbacks=callbacks, compute_metrics=compute_metrics,
        )

        # Train
        resume_from = should_resume(cfg)
        timer.start('training')
        result = run_training(trainer, resume_from=resume_from)
        timer.stop('training')
        print(f"  Training complete: {result}")

        try:
            from evaluation.training_plots import plot_training_progress
            plot_training_progress(cfg=cfg, clearml_task=clearml_task, show=False)
        except Exception as e:
            print(f"  Warning: training plots failed: {e}")

        save_final_model(trainer, cfg)

        # Freeing training memory before evaluation
        import gc
        trainer.optimizer.zero_grad(set_to_none=True)
        if hasattr(trainer, 'get_train_dataloader'):
            try:
                trainer.accelerator.free_memory()
            except Exception:
                pass

        del trainer
        trainer = None
        gc.collect()
        torch.cuda.empty_cache()
        free_gb = (torch.cuda.get_device_properties(0).total_memory
                  - torch.cuda.memory_allocated(0)) / 1e9
        print(f"  GPU memory after cleanup: {free_gb:.1f} GB free")

        # only delete the numbered checkpoint-N directories
        import shutil
        from pathlib import Path

        output_dir = Path(cfg["experiment"]["output_dir"])
        for ckpt in output_dir.glob("checkpoint-[0-9]*"):
            shutil.rmtree(ckpt)
            print(f"  Deleted: {ckpt.name}")

        print(f"  Kept: best_model/      :lowest {cfg['training']['metric_for_best_model']}")
        print(f"  Kept: last_checkpoint/ ")
        print(f"  Kept: final_model/     ")

        # After save_final_model() and checkpoint cleanup, reload best weights:
        from utils.checkpointing import load_best_model_path
        from models.loader import load_base_only
        from transformers import AutoProcessor, AutoModelForImageTextToText

        best_path = load_best_model_path(cfg["experiment"]["output_dir"])
        if best_path:
            print(f"Reloading best model from: {best_path}")
            _device = next(model.parameters()).device

            processor = AutoProcessor.from_pretrained(best_path)
            processor.image_processor.size = {"longest_edge": 512}

            from peft import PeftModel
            if (Path(best_path) / "adapter_config.json").exists():
                # LoRA method: load the correct base for this cfg (no PEFT shell), attach adapter, merge.
                base, _, _ = load_base_only(cfg)
                model = PeftModel.from_pretrained(base, best_path)
                model = model.merge_and_unload()
                from full_component_io import load_non_lora_trainables
                load_non_lora_trainables(model, best_path)
            else:
                # Full method: best_model/ is a complete model.
                model = AutoModelForImageTextToText.from_pretrained(
                    best_path,
                    torch_dtype=model.dtype,
                    trust_remote_code=True,
                    _attn_implementation="eager")

            model.to(_device)
            model.config.use_cache = True

            from verify_weights import verify_trained_weights
            # Compares the reloaded/merged model against a base built for this cfg.
            # full/lora components must differ from base; frozen components must match.
            verify_trained_weights(model, cfg)

            # Pin stop tokens so generate() always has a correct EOS, regardless
            # of what was serialised into best_model/generation_config.json.
            _eos = processor.tokenizer.eos_token_id           # <end_of_utterance>
            _pad = processor.tokenizer.pad_token_id
            model.config.eos_token_id = _eos
            model.config.pad_token_id = _pad
            if getattr(model, "generation_config", None) is not None:
                model.generation_config.eos_token_id = _eos
                model.generation_config.pad_token_id = _pad

            # check model embedding rows match tokenizer length.
            assert model.get_input_embeddings().weight.shape[0] >= len(processor.tokenizer), (
                "Embedding/tokenizer size mismatch after reload, check loader pad setup."
            )

            model.eval()
            print(f"Model loaded best model from: {best_path} "
                  f"(eos_token_id={_eos}, pad_token_id={_pad})")
        else:
            print("WARNING: best_model/ not found, using model currently in memory.")


        from evaluation.report_metrics import run_evaluation_on_dataset
        timer.start('val_evaluation')
        val_results = run_evaluation_on_dataset(
            model=model, processor=processor, dataset=val_ds, cfg=cfg,
            split_name='val', max_new_tokens=cfg['evaluation']['eval_max_new_tokens'],
            batch_size=cfg['evaluation'].get('eval_batch_size', 1),
            compute_groups=["bleu", "rouge", "bertscore", "clinical"],
            clearml_task=clearml_task,
        )

        # Val generation samples
        _run_generation_samples(
            model, processor, val_ds, cfg, N_SAMPLES, timestamp,
            output_dir=Path(cfg['experiment']['output_dir']),
            split='val', device=next(model.parameters()).device,
        )
        timer.stop('val_evaluation')

        # Test set evaluation
        from data.dataset import load_and_prepare_dataset
        active_ds_cfg = (cfg['dataset'] if cfg['dataset_format'] == 'structured'
                         else cfg['csv_dataset'])
        test_path_cfg = active_ds_cfg.get('test_path')
        if not test_path_cfg or '{' in str(test_path_cfg) or not Path(test_path_cfg).is_file():
            raise FileNotFoundError(f'test_path not found: {test_path_cfg}')

        test_ds = load_and_prepare_dataset(
            path=test_path_cfg,
            image_dir=active_ds_cfg['image_dir'],
            dataset_format=active_ds_cfg['format'],
            instruction=active_ds_cfg['instruction'],
            group_images=active_ds_cfg.get('group_images', False),
            max_images_per_sample=active_ds_cfg.get('max_images_per_sample'),
            seed=cfg['experiment']['seed'],
        )
        for s in test_ds:
            s.pop('_image_ref', None)

        timer.start('test_evaluation')
        records = _run_generation_samples(
            model, processor, test_ds, cfg, len(test_ds), timestamp,
            output_dir=Path(cfg['experiment']['output_dir']),
            split='test', device=next(model.parameters()).device,
            all_samples=True,
        )

        # Test metrics
        from evaluation.report_metrics import (
            evaluate_reports, save_metrics, print_metrics_table, metrics_to_clearml)
        predictions = [r['generated'] for r in records]
        references  = [r['reference']  for r in records]

        test_results = evaluate_reports(predictions, references, cfg=cfg,
                                        compute_groups=["bleu", "rouge", "bertscore", "clinical"])
        timer.stop('test_evaluation')

        print_metrics_table(test_results, split_name='Test')
        save_metrics(test_results, cfg, split_name='test')
        if clearml_task:
            metrics_to_clearml(clearml_task, test_results, split_name='test')

        from utils.clearml_integration import log_timing_to_clearml
        log_timing_to_clearml(clearml_task, timer)
        close_task(clearml_task)

        result = subprocess.run([
            "python", f"{REPO_DIR}/tools/ingest_results.py",
            "--exp",              exp_id,
            "--exp-dir",          cfg["experiment"]["output_dir"],
            "--tracker",          TRACKER_PATH,
            "--config",           experiment_config,
            "--timestamp",        timestamp,
            "--clearml-task-id",  clearml_task_id,
            "--gpu",              (torch.cuda.get_device_name(0)
                                   if torch.cuda.is_available() else ""),
            "--exp-name",         exp_name,
            "--overrides",        json.dumps(overrides or {}),

        ], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(f"  Warning: ingest failed: {result.stderr}")

        print(f"\n  {exp_id} COMPLETED successfully.")
        return True

    except Exception as e:
        print(f"\n  {exp_id} FAILED: {e}")
        traceback.print_exc()
        try:
            if clearml_task:
                close_task(clearml_task)
        except Exception:
            pass
        return False

    finally:
        # clean up GPU memory so the next experiment starts fresh
        _cleanup_gpu(model=model, trainer=trainer)


# generate sample reports and write to disk
def _run_generation_samples(model, processor, dataset, cfg, n_samples, timestamp,
                             output_dir, split, device, all_samples=False):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if all_samples:
        samples  = dataset
        indices  = list(range(len(dataset)))
    else:
        rng     = random.Random(cfg['experiment']['seed'])
        indices = rng.sample(range(len(dataset)), k=min(n_samples, len(dataset)))
        samples = [dataset[i] for i in indices]

    max_new_tokens = cfg['evaluation'].get('eval_max_new_tokens',
                                       cfg['evaluation']['max_new_tokens'])
    model.eval()
    records = []

    for idx, sample in enumerate(samples):
        messages  = sample['messages']
        user_turn = [messages[0]]
        content   = messages[1]['content']
        reference = content[0]['text'] if isinstance(content, list) else content

        try:
            imgs = sample['images']
            if not isinstance(imgs, (list, tuple)):
                imgs = [imgs]
            text = processor.apply_chat_template(
                user_turn, add_generation_prompt=True, tokenize=False)
            inputs = processor(text=[text], images=[imgs], return_tensors='pt', padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    eos_token_id=processor.tokenizer.eos_token_id,
                    pad_token_id=processor.tokenizer.pad_token_id,
                )
            input_len = inputs['input_ids'].shape[1]
            generated = processor.tokenizer.decode(
                generated_ids[0][input_len:], skip_special_tokens=True)
        except Exception as e:
            generated = f"[GENERATION FAILED: {e}]"

        records.append({
            'sample_idx': idx + 1,
            'val_index':  indices[idx] if not all_samples else idx,
            'n_images':   len(sample['images']),
            'reference':  reference,
            'generated':  generated,
        })

    model.train()

    txt_path  = output_dir / f'{split}_generations.txt'
    json_path = output_dir / f'{split}_generations.json'

    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(f"Experiment : {cfg['experiment']['name']}\n")
        f.write(f"Timestamp  : {timestamp}\nSplit: {split}\n{'='*60}\n\n")
        for r in records:
            f.write(f"Sample {r['sample_idx']} | images: {r['n_images']}\n")
            f.write(f"REFERENCE:\n{r['reference']}\n{'-'*60}\n")
            f.write(f"GENERATED:\n{r['generated']}\n\n")

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump({
            'experiment': cfg['experiment']['name'],
            'timestamp': timestamp, 'split': split,
            'generations': records,
        }, f, indent=2, ensure_ascii=False)

    return records


# RUN THE QUEUE
results_log = []

print(f"{'='*70}")
print(f"  EXPERIMENT QUEUE: {len(EXPERIMENT_QUEUE)} experiment(s) to run")
for i, exp in enumerate(EXPERIMENT_QUEUE, 1):
    print(f"  {i}. {exp['exp_id']:12} — {exp['description']}")
print(f"{'='*70}\n")

for i, exp_cfg in enumerate(EXPERIMENT_QUEUE):
    exp_id      = exp_cfg['exp_id']
    config_path = exp_cfg['config_path']
    description = exp_cfg.get('description', '')
    overrides   = exp_cfg.get('overrides', {})

    queue_remaining = len(EXPERIMENT_QUEUE) - i - 1

    success = run_experiment(
        exp_id=exp_id,
        config_path=config_path,
        description=description,
        overrides=overrides,
    )

    results_log.append({'exp_id': exp_id, 'success': success})

    if queue_remaining > 0:
        print(f"\n  Next up: {queue_remaining} experiment(s) remaining.")
        print(f"  GPU cooldown: 30 seconds...\n")
        import time
        time.sleep(30)   # brief cooldown between experiments


print(f"\n{'='*70}")
print("  ALL EXPERIMENTS COMPLETE")
print(f"{'='*70}")
for r in results_log:
    status = "SUCCESS" if r['success'] else "FAILED"
    print(f"  {r['exp_id']:12}  {status}")
print(f"{'='*70}\n")
print("Results have been ingested into the tracker.")


In [ ]:
# Release Colab GPU runtime
from google.colab import runtime
runtime.unassign()
